In [2]:
import requests
import json

以下示例代码演示如何使用 `request_qianfan_endpoint` 进行图表理解：
1. **API 密钥和端点**：定义了 `openai_api_key` 和 `openai_endpoint` 以访问 API。
2. **文本提示 (`user_prompt`)**：定义了一个文本提示，在此示例中，要求模型“介绍一下图片”。
3. **图像 URL (`images`)**：提供了一个包含一个或多个图像 URL 的列表。这些是模型将分析的图像。
4. **函数调用**：使用端点、提示、API 密钥、模型名称 (`qianfan-ocr`)、最大tokens和 `image_urls` 列表调用 `request_qianfan_endpoint` 函数。
5. **响应处理**：然后打印 API 的 JSON 响应，让您可以看到模型根据多模态输入输出的结果。


In [16]:
import requests
from typing import Optional, List

def request_qianfan_endpoint(
    api_endpoint: str,
    prompt_text: str,
    api_key: str,
    model: str = "gpt-3.5-turbo",
    temperature: float = 0.7,
    max_tokens: int = 150,
    image_urls: Optional[List[str]] = None
) -> dict:
    """
    Sends a request to an OpenAI-style API endpoint, supporting multimodal input.

    Args:
        api_endpoint (str): The URL of the API endpoint (e.g., 'https://api.openai.com/v1/chat/completions').
        prompt_text (str): The text prompt for the model.
        api_key (str): Your API key.
        model (str): The model identifier to use (default: 'gpt-3.5-turbo').
        temperature (float): Controls randomness. Lower values make output more deterministic (default: 0.7).
        max_tokens (int): The maximum number of tokens to generate in the completion (default: 150).
        image_urls (Optional[List[str]]): A list of image URLs to include in the prompt for multimodal models.

    Returns:
        dict: The JSON response from the API.
    """
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    messages_content = []
    if prompt_text:
        messages_content.append({"type": "text", "text": prompt_text})

    if image_urls:
        for url in image_urls:
            messages_content.append({"type": "image_url", "image_url": {"url": url}})

    if not messages_content:
        raise ValueError("Either prompt_text or image_urls must be provided.")

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": messages_content if len(messages_content) > 1 or image_urls else prompt_text
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    try:
        response = requests.post(api_endpoint, headers=headers, json=payload)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response content: {e.response.text}")
        return {"error": str(e)}


In [17]:
# Example usage:
openai_api_key = ""
model = ""
openai_endpoint = "https://qianfan.baidubce.com/v2/chat/completions"

### 样例一： 图表caption


![image](https://img.freepik.com/premium-vector/illustration-trend-bar-chart_94753-320.jpg)

In [27]:
user_prompt = "介绍一下以下这张图表"
images = ["https://img.freepik.com/premium-vector/illustration-trend-bar-chart_94753-320.jpg"]

response_data = request_qianfan_endpoint(openai_endpoint,
                                                user_prompt,
                                                openai_api_key,
                                                model=model,
                                                max_tokens=256,
                                                image_urls=images)

print(json.dumps(response_data, ensure_ascii=False, indent=2))

{
  "id": "as-wyex5tqbj8",
  "object": "chat.completion",
  "created": 1773232072,
  "model": "tmpz1seg_qfocrfp8:amv-s0iqjxa4hmn4",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The image shows a bar chart titled \"Trend Bar Chart Values with categories.\" The chart is a stacked bar chart that displays values over a series of years from 2014 to 2030. The vertical axis is labeled \"Value\" and the horizontal axis represents the \"Years.\" There are four distinct categories, each represented by a different color, with a legend at the bottom of the chart indicating the category titles. The colors used are blue, green, orange, and red, corresponding to the categories.\n\nThe bars increase in height over the years, indicating a rising trend. The values for each year are labeled on top of the bars, starting at 56 in 2014 and increasing to 1200 in 2030. The chart also includes a green upward arrow graphic at the top right, suggestin

样例二： 图表信息结构化输出

![image](https://img.freepik.com/premium-vector/illustration-trend-bar-chart_94753-320.jpg)

In [26]:
user_prompt = """任务目标：请将图片中的统计图表解析成字典形式，输出为json格式。
输出要求：
    1. 提取图表中的信息组织为字典格式。
    2. 严格按照输出格式输出，不要输出额外其他内容。
输出格式
```json
xxx
```"""
images = ["https://img.freepik.com/premium-vector/illustration-trend-bar-chart_94753-320.jpg"]

response_data = request_qianfan_endpoint(openai_endpoint,
                                                user_prompt,
                                                openai_api_key,
                                                model=model,
                                                max_tokens=256,
                                                image_urls=images)

print(json.dumps(response_data, ensure_ascii=False, indent=2))

{
  "id": "as-zq845ma1a3",
  "object": "chat.completion",
  "created": 1773232051,
  "model": "tmpz1seg_qfocrfp8:amv-s0iqjxa4hmn4",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "```json\n{\n  \"title\": \"Trend Bar Chart Values\",\n  \"subtitle\": \"with categories\",\n  \"source\": \"Data Source\",\n  \"x_title\": \"Years\",\n  \"y_title\": \"Value\",\n  \"values\": {\n  \"Category title\": {\n    \"2014\": \"56\",\n    \"2015\": \"128\",\n    \"2016\": \"216\",\n    \"2017\": \"180\",\n    \"2018\": \"320\",\n    \"2019\": \"480\",\n    \"2020\": \"515\",\n    \"2021\": \"650\",\n    \"2022\": \"1000\",\n    \"2023\": \"1200\"\n  },\n  \"Category title\": {\n    \"2014\": \"56\",\n    \"2015\": \"128\",\n    \"2016\": \"216\",\n    \"2017\": \"180\",\n    \"2018\": \"320\",\n    \"2019\": \"48"
      },
      "finish_reason": "length",
      "flag": 0
    }
  ],
  "usage": {
    "prompt_tokens": 330,
    "completion_tokens"

样例三：图表QA与预测
![image](https://substackcdn.com/image/fetch/$s_!tGPG!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F78686590-9246-4c71-aa55-41525cdcc062_1666x1484.png)

In [28]:
user_prompt = "下面的这张图表里能得出什么结论？怎么看待2025-2030的走势"

images= ["https://substackcdn.com/image/fetch/$s_!tGPG!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F78686590-9246-4c71-aa55-41525cdcc062_1666x1484.png"]
response_data = request_qianfan_endpoint(openai_endpoint,
                                                user_prompt,
                                                openai_api_key,
                                                model=model,
                                                max_tokens=1024,
                                                image_urls=images)

print(json.dumps(response_data, ensure_ascii=False, indent=2))

{
  "id": "as-tdpn99vhgq",
  "object": "chat.completion",
  "created": 1773232193,
  "model": "tmpz1seg_qfocrfp8:amv-s0iqjxa4hmn4",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "这张图表展示了美国大型公司股票的总回报率，采用对数刻度显示。从1824年到2024年，图表清晰地显示了一个长期上升的趋势。以下是我从图表中得出的几点结论：\n\n1. 长期趋势：过去200年来，美国大型公司股票价格总体上呈上升趋势，没有持续下跌的时期。\n\n2. 历史波动：虽然整体趋势向上，但每个时期的收益率都有波动，特别是在1920年代和1930年代出现了明显下跌。\n\n3. 2020年代表现：2000年代以来，股票市场经历了持续上涨，近年来增速有所放缓但仍然保持上升。\n\n关于2025-2030年的走势预测：\n\n1. 基于图表的历史模式，未来5-10年很可能会继续呈现上升趋势，尽管增速可能不会像过去那样快。\n\n2. 未来走势可能受到多种因素影响，包括全球经济状况、政策变化、技术进步和市场周期等。\n\n3. 持续的长期上升趋势表明，从长期来看，股票市场表现通常是积极的，但这不代表短期内不会出现大幅波动。\n\n4. 投资者应记住，虽然历史趋势具有参考价值，但未来的市场走势仍然受到多重复杂因素的影响，需要结合其他分析工具和当前经济环境进行综合判断。"
      },
      "finish_reason": "stop",
      "flag": 0
    }
  ],
  "usage": {
    "prompt_tokens": 2594,
    "completion_tokens": 293,
    "total_tokens": 2887
  }
}
